# GTEx tissue feature importance analysis (true GTEx SMTSD labels + SHAP)

This report reads Snakemake production artifacts only. The nested-CV RandomForest + SHAP training is done by `scripts/gtex/lv_importance_true_labels.py` (rule `lv_importance_rf_true_labels_gtex`). This notebook loads the LV matrix for a correlation sanity check and displays the script's output tables.

💡 **Environment:** `clamp-analyses`

## Libraries

In [ ]:
import csv
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import rpy2.robjects as ro
import seaborn as sns
from pyprojroot.here import here

# Settings

In [ ]:
OUTPUT_DIR = here('output/03_model_biology/01_gtex/01_LV_importance_rf_true_labels/gtex_feature_importance_true_labels_binary_shap')

print(f"Output directory: {OUTPUT_DIR}")

# Output

In [ ]:
sys.path.insert(0, str(here("scripts/gtex")))
from common import extract_B_matrix, extract_summary_matrix

readRDS = ro.r["readRDS"]

# load .rds files from CLAMP with different priors
gtex_CLAMPfull_rds = readRDS(str(here('output/01_model_building/01_gtex/CLAMPfull.rds')))
gtex_CLAMPfull = extract_B_matrix(gtex_CLAMPfull_rds)

lv_data = gtex_CLAMPfull
lv_data.head()

# Load data

In [ ]:
path = here('data/gtex/GTEx_Analysis_v8_Annotations_SampleAttributesDS.txt')

gtex_meta = pd.read_csv(
    path,
    sep='\t',
    header=0,
    dtype=str,
    quoting=csv.QUOTE_NONE,
    engine='python',
    comment=None,
    keep_default_na=False,
    on_bad_lines='warn'
)

print(gtex_meta.shape)
gtex_meta.head()

In [ ]:
gtex_CLAMPfull_summary = extract_summary_matrix(gtex_CLAMPfull_rds)
gtex_CLAMPfull_summary["pathway"] = gtex_CLAMPfull_summary["pathway"].str.replace("C2CP_", "", regex=False)

print(gtex_CLAMPfull_summary.shape)
gtex_CLAMPfull_summary.head()


In [ ]:
# samples x LVs
lv_matrix = lv_data.T

# correlation matrix
corr_matrix = lv_matrix.corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0,
            xticklabels=False, yticklabels=False)
plt.show()

# find highly correlated pairs
high_corr_threshold = 0.9
high_corr_pairs = []
for i in range(len(corr_matrix.columns)):
    for j in range(i+1, len(corr_matrix.columns)):
        if abs(corr_matrix.iloc[i, j]) > high_corr_threshold:
            high_corr_pairs.append((
                corr_matrix.columns[i],
                corr_matrix.columns[j],
                corr_matrix.iloc[i, j]
            ))

print(f"Number of highly correlated pairs (|r| > {high_corr_threshold}): {len(high_corr_pairs)}")

# Feature importance results (RandomForest binary + SHAP)

Computed by `scripts/gtex/lv_importance_true_labels.py` (rule `lv_importance_rf_true_labels_gtex`) and read here for inspection.

In [ ]:
accuracy_df = pd.read_csv(OUTPUT_DIR / "accuracy_summary.tsv", sep="\t")
shap_results_df = pd.read_csv(OUTPUT_DIR / "all_shap_positive.tsv", sep="\t")
cumulative_df = pd.read_csv(OUTPUT_DIR / "cumulative_importance.tsv", sep="\t")


In [ ]:
accuracy_df.sort_values(by='Mean_CV_Accuracy', ascending=False)


In [ ]:
cumulative_df.head(40)
